# Appendix D: Building an Agent

Assemble the course into one working loop: routing, tools, memory, and an eval harness.

This notebook is an original tutorial (rewritten to deliver its title: an
end-to-end mini agent, built from the patterns of notebooks 02, 05, 08, 19).

## Learning Objectives

- Assemble the canonical agent loop: observe → decide (route) → act (tool) → observe result → repeat → answer.
- Wire in the four subsystems: router, tool registry, memory, termination.
- Attach a minimal eval harness and run it against the agent.
- Know what separates this toy from production (the upgrade path through the rest of the course).

## Mental Model

Every agent — Claude Code, a voice interviewer, a support bot — is this loop
with better parts:

```
        ┌────────────────────────────────────┐
        ▼                                    │
user msg ─▶ [context: system + memory + history]
                 │
                 ▼
              model decides ──▶ tool call ──▶ execute ──▶ append observation ─┘
                 │
                 ▼ (no tool needed, or budget hit)
              final answer ──▶ eval harness scores it
```

Build the skeleton once and every course notebook becomes a part upgrade:
better routing (02), better tools (05), better memory (08), guardrails (18),
evals (19/22), state graph + checkpoints when the loop needs durability (28).

## Important Concepts

- **The loop owns control flow, the model owns decisions.** Your code decides *that* there is a step budget; the model decides *what* to do with each step.
- **Termination**: three exits — model answers, step budget exhausted, error escalation. Every production incident involves a missing exit.
- **Tool registry**: schema + validate + execute per tool (notebook 05's contract).
- **Memory injection**: relevant memories retrieved and labeled in context each turn (notebook 08).
- **Eval harness**: golden cases with property assertions run against the whole agent, not just the model (notebook 19/22).

## Need-To-Know Coverage Checklist

- [x] The full agent loop in runnable code — observe, decide, act, terminate.
- [x] Router, tool registry, memory, and evals wired together.
- [x] Step budgets and error escalation.
- [x] The production upgrade path, mapped to course notebooks.

## Deep Study Notes

### What the toy deliberately omits (the upgrade path)

| Toy shortcut | Production part | Notebook |
|---|---|---|
| keyword router | LLM/embedding router with confidence | 02, 30 |
| dict tool registry | MCP servers, scoped per agent | 05, 10, 28 |
| list memory | compaction + vector retrieval | 08, 25 |
| in-process state | typed state graph + checkpoints | 28 |
| 3 golden cases | versioned dataset + judges + CI gate | 19, 22 |
| no guardrails | input/output rails, injection defense | 18 |

### Design decisions the mini agent still makes correctly

- Errors become observations, not crashes.
- The loop is bounded.
- Memory is injected as labeled context, not silently mixed into history.
- The eval harness exercises the *agent* end to end — router bugs and tool
  bugs fail evals even when the "model" is fine. Evaluating components in
  isolation misses integration failures, which are most failures.

## Common Failure Modes

- No step budget → tool loops forever on a confusing observation.
- Evaluating only the model, not the loop → integration bugs ship.
- Memory dumped unlabeled into history → the model can't distinguish facts from conversation.
- One giant "do everything" system prompt instead of routed behaviors.
- Toy shipped as production because it demos well — the table above is the honest gap list.

## Implementation Notes

- Keep the loop pure: state in, state out — it makes the eval harness trivial and the checkpoint upgrade (28) natural.
- Log every decision (route, tool, args) from day one.
- Write the eval cases before adding the next feature — EDD applies to agents, not just prompts.

In [ ]:
"""A complete mini agent: router + tools + memory + loop + eval harness.

Every part is the honest toy version of a course pattern; swap parts to
upgrade. Runs standalone.
"""

# --- parts -----------------------------------------------------------------
MEMORY = [{"text": "user's company: AcmeCo", "tags": {"company", "acmeco"}}]

TOOLS = {
    "invoice_lookup": {
        "validate": lambda a: None if a.get("id") else "id required",
        "execute": lambda a: {"id": a["id"], "status": "overdue", "amount": 420},
    },
}


def recall(msg):
    words = set(msg.lower().split())
    return [m["text"] for m in MEMORY if m["tags"] & words]


def route(msg):
    if "invoice" in msg.lower():
        return "billing"
    return "general"


def fake_model(context, step):
    """Scripted decisions: ask for a tool on step 0, answer on step 1."""
    if context["route"] == "billing" and step == 0:
        return {"tool": "invoice_lookup", "args": {"id": "INV-7"}}
    obs = context["observations"]
    memo = f" (context: {context['memories'][0]})" if context["memories"] else ""
    return {"answer": f"Invoice INV-7 is overdue, $420.{memo}" if obs
            else "How can I help?"}


# --- the loop ----------------------------------------------------------------
def run_agent(msg, max_steps=4):
    context = {"route": route(msg), "memories": recall(msg), "observations": []}
    for step in range(max_steps):
        decision = fake_model(context, step)
        if "answer" in decision:
            return {"answer": decision["answer"], "steps": step,
                    "route": context["route"]}
        tool = TOOLS[decision["tool"]]
        err = tool["validate"](decision["args"])
        obs = {"error": err} if err else tool["execute"](decision["args"])
        context = {**context, "observations": context["observations"] + [obs]}
    return {"answer": "I hit my step budget - escalating to a human.",
            "steps": max_steps, "route": context["route"]}


# --- eval harness: golden cases with property assertions --------------------
CASES = [
    {"msg": "Is my acmeco invoice paid?",
     "props": [lambda r: r["route"] == "billing",
               lambda r: "overdue" in r["answer"],
               lambda r: "AcmeCo" in r["answer"]]},   # memory was used
    {"msg": "hello there",
     "props": [lambda r: r["route"] == "general", lambda r: r["steps"] == 0]},
]

for case in CASES:
    result = run_agent(case["msg"])
    passed = sum(p(result) for p in case["props"])
    print(f"[{passed}/{len(case['props'])}] {case['msg']!r} -> {result['answer']}")


## Practice

1. Break the router (make "invoice" route to general) and rerun the evals —
   note that the harness catches it even though the model and tools are fine.
2. Add a second tool and a case where the model must call both before answering.
3. Add the missing exit: a tool that always errors, and an escalation after
   two consecutive error observations.
4. Pick any row of the upgrade table and sketch the swap: what interface stays
   the same, what changes behind it?

## Design Checklist

- [ ] Loop bounded; all three exits implemented (answer, budget, escalation).
- [ ] Tool errors become observations.
- [ ] Memory injected labeled; retrieval relevance-filtered.
- [ ] Evals exercise the assembled agent, not components in isolation.
- [ ] Every decision logged (route, tool, args, steps).
- [ ] The toy-to-production gap list is explicit and tracked.